# Super Deep MLP!!

In [ ]:
!nvidia-smi

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
from torch import nn

from idl.common import ClassifierTrainer, TensorboardLogger, count_parameters, get_datasets_and_loaders
from idl.week1.analysis import plot_learning_curves

## Data

In [ ]:
batch_size = 128
train_data, test_data, train_dataloader, test_dataloader = get_datasets_and_loaders("mnist", batch_size=batch_size, num_workers=8, verbose=False)

## Model

Woah, 8 hidden layers!

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

n_layers = 8
hidden_dim = 128
input_dim = 784
model = nn.Sequential(nn.Flatten())
for ind in range(n_layers):
    model.append(nn.Linear(input_dim if ind == 0 else hidden_dim, hidden_dim))
    model.append(nn.ReLU())
model.append(nn.Linear(hidden_dim, 10))

print(model)

# intialize BIG MODEL with BIG NUMBERS!!
def _init(layer: nn.Module):
    if isinstance(layer, nn.Linear):
        nn.init.uniform_(layer.weight, -0.4, 0.4)


with torch.no_grad():
    model.apply(_init)

model = model.to(device)

In [ ]:
n_params = count_parameters(model)
print(f"Model with {n_params:,d} parameters.")

## Training

The important change is the new `logger` object. Some NOTES:
- You might want to empty the log directory after each run (or rather, before starting a new one with the same directory), or take care to choose a new directory for each run. For example, `fail1_run0`, `fail1_run1`, etc.
  - Advanced exercise: Automate this process of creating new numbered "versions" :)
- In case things go wrong in training very early, you can reduce `step_frequency` (extrem case: `step_frequency=1` and train for just a single epoch. This can give you a better view of what happens when.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

logger = TensorboardLogger("logs/fail1", step_frequency=100)

trainer = ClassifierTrainer(model=model, optimizer=optimizer, logger=logger,
                            training_loader=train_dataloader, validation_loader=test_dataloader,
                            n_epochs=40, device=device)

In [ ]:
metrics = trainer.train_model()

In [ ]:
plot_learning_curves(metrics, keys=["cross_entropy", "accuracy"])

In [ ]:
trainer.plot_examples()

## Tensorboard within notebook

In [ ]:
%load_ext tensorboard

In [ ]:
# NOTE!! In case you are running this on a local machine, this port must be unused.
# you also cannot run multiple TB instances on one port!
# but in general, having just one instance running on the log directory should be sufficient.
%tensorboard --logdir=logs --port=6006